In [7]:
import random
import numpy as np
import copy
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import matplotlib.patches as patches
from collections import defaultdict

from layouts import keys_dvorak, keys_qwerty, characters_qwerty

def get_key_position(key, keys):
    """Returns the (x, y) position of a key from the 'keys' dictionary."""
    return keys[key]['pos']   

def calculate_distance(pos1, pos2):
    """Calculate the Euclidean distance between two points (pos1 and pos2)."""
    return np.sqrt((pos2[0] - pos1[0])**2 + (pos2[1] - pos1[1])**2)

def analyze_text_and_calculate_distance(text, keys, characters):
    """
    Analyze the given text and calculate the total finger travel distance on the keyboard.
    
    Args:
        text: The text to analyze.
        keys: The keyboard layout and positions.
        characters: The character-to-keystroke mapping.
    
    Returns:
        total_distance: The total finger travel distance.
        key_counts: A dictionary with the frequency each key was pressed.
    """
    key_counts = defaultdict(int)
    total_distance = 0
    for char in text:
        if char in characters:  # Check if the character has a corresponding keystroke
            sequence = characters[char]  # Get the key sequence
            for i, key in enumerate(sequence):
                key_pos = get_key_position(key, keys)
                start_pos = get_key_position(keys[key]['start'], keys)
                distance = calculate_distance(start_pos, key_pos)  # Calculate distance between start and key positions
                total_distance += distance
                key_counts[key] += 1  # Count the key press frequency
    return total_distance, key_counts


In [8]:
def get_neighbour(layout):
    """
    Generate a neighboring keyboard layout by swapping two keys.

    Args:
        layout: The current keyboard layout.

    Returns:
        new_layout: A new keyboard layout with two keys swapped.
    """
    new_layout = copy.deepcopy(layout)
    fixed_keys = {'Ctrl_L', 'Ctrl_R', 'Alt_L', 'Alt_R', 'Space', 'Shift_L', 'Shift_R', 'Win', 'Fn', 'Enter', 'Backspace', 'CapsLock', 'Tab'}
    home_row_keys = {'a', 's', 'd', 'f', 'g', 'h', 'j', 'k', 'l', ';', "'"}
    
    # Identify keys that can be moved
    movable_keys = [key for key in new_layout if key not in fixed_keys and not key.isdigit()]
    
    # Randomly select two keys to swap
    key1, key2 = random.sample(movable_keys, 2)
    
    # Swap positions of the selected keys
    new_layout[key1]['pos'], new_layout[key2]['pos'] = new_layout[key2]['pos'], new_layout[key1]['pos']
    
    # Update the start key if home row keys are swapped
    if key1 in home_row_keys or key2 in home_row_keys:
        for key in new_layout:
            if new_layout[key]['start'] == key1:
                new_layout[key]['start'] = key2
            elif new_layout[key]['start'] == key2:
                new_layout[key]['start'] = key1
    
    return new_layout


In [9]:
def simulated_annealing(keys, text, initial_temp, cooling_rate, num_iterations):
    """
    Perform simulated annealing to optimize the keyboard layout.

    Args:
        keys: The initial keyboard layout.
        text: The text to analyze.
        initial_temp: The initial temperature for the annealing process.
        cooling_rate: The rate at which the temperature decreases.
        num_iterations: The number of iterations to perform.

    Returns:
        best_layout: The optimized keyboard layout.
        best_distance: The total finger travel distance for the optimized layout.
        distances: A list of distances recorded at each iteration.
    """
    # Initialize the current layout and distance
    current_layout = copy.deepcopy(keys)
    current_distance, _ = analyze_text_and_calculate_distance(text, current_layout, characters_qwerty)
    best_layout = copy.deepcopy(current_layout) 
    best_distance = current_distance
    distances = [current_distance]
    temp = initial_temp

    # Perform simulated annealing
    for i in range(num_iterations):
        # Generate a neighboring layout
        neighbour_layout = get_neighbour(current_layout)
        neighbour_distance, _ = analyze_text_and_calculate_distance(text, neighbour_layout, characters_qwerty)
        
        # Calculate the difference in distance
        diff = current_distance - neighbour_distance
        # Calculate the acceptance probability
        p = np.exp(diff / temp)
        
        # Accept the new layout if it improves the distance or with a certain probability
        if diff > 0 or random.random() < p:
            current_layout = copy.deepcopy(neighbour_layout)
            current_distance = neighbour_distance
            # Update the best layout if the current layout is better
            if current_distance < best_distance:
                best_layout = copy.deepcopy(current_layout)
                best_distance = current_distance
        
        # Decrease the temperature
        temp *= cooling_rate
        distances.append(current_distance)
    
    return best_layout, best_distance, distances


In [10]:
def plot_keyboard(layout, title='Keyboard Layout'):
    """
    Plots the keyboard layout.

    Args:
        layout: The keyboard layout dictionary with key positions.
        title: The title of the plot.
    """
    fig, ax = plt.subplots(figsize=(15, 6))
    for key, data in layout.items():
        x, y = data['pos']
        rect = Rectangle((x-0.4, y-0.4), 0.8, 0.8, fill=False)
        ax.add_patch(rect)
        ax.text(x, y, key, ha='center', va='center', fontsize=12)
    ax.set_xlim(-1, 15)
    ax.set_ylim(-1, 5)
    ax.set_aspect('equal', adjustable='box')
    ax.axis('off')
    plt.title(title)
    plt.tight_layout()
    plt.show()

def plot_cost(costs):
    """
    Plots the distance cost over iterations.

    Args:
        costs: A list of distance costs recorded at each iteration.
    """
    plt.plot(costs)
    plt.xlabel('Iteration')
    plt.ylabel('Distance')
    plt.title('Distance vs. Iteration')
    plt.show()

In [11]:
def draw_key(ax, x, y, width, height, key, color, shadow_offset, shifted_symbol=None):
    """Draws a single key with optional shifted symbol."""
    # Draw the shadow for the key
    ax.add_patch(patches.Rectangle(
        (x + shadow_offset, y - shadow_offset), width, height, edgecolor='none', facecolor='gray', alpha=0.5))
    # Draw the key rectangle
    ax.add_patch(patches.Rectangle((x, y), width, height, edgecolor='black', facecolor=color, lw=2))
    # Add the key label in the center
    plt.text(x + width / 2, y + height / 2, key, ha='center', va='center', fontsize=12, color='black')
    # If the key has a shifted symbol, display it at the top-right
    if shifted_symbol:
        plt.text(x + width - 0.2, y + height - 0.2, shifted_symbol, ha='right', va='top', fontsize=8, color='black')

def draw_heatmap_rectangle(ax, x, y, width, height, color, alpha):
    """Draws the heatmap rectangle over the keys."""
    ax.add_patch(patches.Rectangle((x, y), width, height, edgecolor='none', facecolor=color, alpha=alpha))

def draw_heatmap(keys, key_counts, characters):
    """Draws the keyboard layout and heatmap based on the key press frequencies.
    
    Args:
        keys: The keyboard layout and positions.
        key_counts: The frequency each key was pressed.
        characters: The character-to-keystroke mapping.
    """
    fig, ax = plt.subplots(figsize=(20, 8))
    fig.patch.set_facecolor('#2e2e2e')  # Set background color
    ax.set_facecolor('#2e2e2e')
    
    key_width, key_height, spacing = 1, 1, 0.3  # Define dimensions and spacing between keys
    shadow_offset = 0.1  # Offset for key shadows

    max_count = max(key_counts.values(), default=1)  # Get the max count to normalize the color intensity
    norm = plt.Normalize(0, max_count)
    cmap = plt.cm.plasma  # Color map for the heatmap

    # Iterate through the keys to draw each key on the keyboard
    for key, data in keys.items():
        x, y = data['pos']
        x_pos, y_pos = x * (key_width + spacing), y * (key_height + spacing)
        
        width = key_width  # Adjust width for special keys        
        shifted_symbol = None  # Handle shifted symbols
        if not key.isalpha():
            for char, sequence in characters.items():
                if len(sequence) == 2 and sequence[1] == key:
                    shifted_symbol = char
                    break
        
        # Draw the key with its shadow and possible shifted symbol
        draw_key(ax, x_pos, y_pos, width, key_height, key, '#f0f0f0', shadow_offset, shifted_symbol)
        
        if key in key_counts and key != 'Space':  # Draw heatmap for pressed keys
            color = cmap(norm(key_counts[key]))
            draw_heatmap_rectangle(ax, x_pos, y_pos, width, key_height, color, 0.8)

    ax.set_xlim(-1, 20)
    ax.set_ylim(-1, 8)
    plt.axis('off')  # Turn off axis for cleaner visualization
    plt.title("Keyboard Layout Heatmap", color='white')

    # Save the figure to a file
    plt.savefig("keyboard_heatmap.png", bbox_inches='tight', facecolor='#2e2e2e')
    
    # Show the heatmap
    plt.show()


In [12]:
text_to_analyze = "Store Layouts and Distances: The layouts_and_distances dictionary stores all the layouts and their corresponding distances for each run.Track Best Layout: The function tracks the best layout and its distance during multiple runs of simulated annealing.Calculate Average Distance: The average distance is calculated by summing all the distances and dividing by the number of runs."
initial_temp = 100
cooling_rate = 0.999
num_iterations = 8000
num_runs = 10

def optimized_keyboard(keys):
    """
    Optimize the keyboard layout using simulated annealing and visualize the results.

    Args:
        keys: The initial keyboard layout can be either 'keys_qwerty' or 'keys_dvorak'.

    Returns:
        best_layout: The optimized keyboard layout with the least travel distance.
    """
    # Calculate the initial total finger travel distance and key press frequencies
    total_finger_distance, key_counts = analyze_text_and_calculate_distance(text_to_analyze, keys, characters_qwerty)
    print(f"Original total finger travel distance: {total_finger_distance:.2f} units")
    # Draw the initial heatmap of the keyboard layout
    draw_heatmap(keys, key_counts, characters_qwerty)
    
    best_layout = None
    best_distance = float('inf')
    best_distances = []
    layouts_and_distances = {}
    
    # Perform multiple runs of simulated annealing to find the best layout
    for run in range(num_runs):
        layout, distance, distances = simulated_annealing(keys, text_to_analyze, initial_temp, cooling_rate, num_iterations)
        layouts_and_distances[run] = (layout, distance)
        if run == 0 or distance < best_distance:
            best_layout = copy.deepcopy(layout)
            best_distance = distance
            best_distances = distances
        print(f"Run {run+1}: Total finger travel distance: {distance:.2f} units")

    # Calculate the average distance
    average_distance = sum(distance for _, distance in layouts_and_distances.values()) / num_runs
    print(f"Average distance over {num_runs} runs: {average_distance:.2f} units")
    
    # Plot the cost (distance) over iterations
    plot_cost(best_distances)

    if keys == keys_dvorak:
        if total_finger_distance - average_distance < 30:
            best_distance = total_finger_distance
            best_layout = keys_dvorak
    
    # Calculate the total finger travel distance for the optimized layout
    total_finger_distance_optimized, key_counts_optimized = analyze_text_and_calculate_distance(text_to_analyze, best_layout, characters_qwerty)

    # Draw the heatmap of the optimized keyboard layout
    draw_heatmap(best_layout, key_counts_optimized, characters_qwerty)

    
    # Print the total finger travel distances before and after optimization
    print(f"Total finger travel distance before optimization: {total_finger_distance:.2f} units")
    print(f"Final total finger travel distance after optimization: {total_finger_distance_optimized:.2f} units")
  

In [ ]:
optimized_keyboard(keys_qwerty)

In [ ]:
optimized_keyboard(keys_dvorak)